In [ ]:
import pandas as pd
import matplotlib as plt
import json
from datetime import datetime, timedelta
df_fall_26 = pd.read_csv('output/fall_26.csv')
df_spring_26 = pd.read_csv('output/spring_26.csv')
df_fall_25 = pd.read_csv('output/fall_25_new_scrape.csv')

desired_codes = ['SEAS', 'SAS'] 

df_fall_26
df_spring_26
df_fall_25
#group class times

def group_class(df, term):
    # drop rows with missing days/time
    df = df.dropna(subset=['days'])
    df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
    df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

    df['days'] = df['days'].str.split().str[0]


    day_map = {
        'M': 'Monday',
        'T': 'Tuesday',
        'W': 'Wednesday',
        'R': 'Thursday',
        'F': 'Friday'
    }
    df['days'] = df['days'].fillna('').astype(str)

    df['days'] = df['days'].apply(
        lambda x: [day_map.get(d) for d in x if d in day_map]
    )
    df = df.explode('days').reset_index(drop=True)

    time_bins = pd.date_range("08:00", "23:00", freq="30min").time

    rows = []

    for _, row in df.iterrows():
        day = row['days']
        start = row['start_time'].time()
        end = row['end_time'].time()

        for t in time_bins:
            # convert time → minutes since midnight
            t_minutes = t.hour * 60 + t.minute
            bin_start_minutes = t_minutes
            bin_end_minutes = t_minutes + 30

            start_minutes = start.hour * 60 + start.minute
            end_minutes = end.hour * 60 + end.minute

            # overlap condition
            if start_minutes < bin_end_minutes and end_minutes > bin_start_minutes:
                rows.append({
                    'Day': day,
                    'Time': t
                })

    binned_df = pd.DataFrame(rows)
    # binned_df.to_csv(f"filtered_{term}.csv")
    result = (
        binned_df
        .groupby(['Day', 'Time'])
        .size()
        .reset_index(name='Share')
    )

    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
    result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

    result = result.sort_values(['Day', 'Time'])

    result.to_csv(f'{term}_testing.csv', index=False)

    return result

desired_type = ['-REC', '-DISC', 'recitation', 'discussion', 'lab']
pattern = '|'.join(desired_type)

fall_filtered_df = df_fall_26[df_fall_26['school_code'].isin(desired_codes)]
display(fall_filtered_df.shape[0])

fall_filtered_df = fall_filtered_df[~fall_filtered_df['course_name'].str.contains(pattern, case=False, na=False)]
display(fall_filtered_df.shape[0])

display(fall_filtered_df)

fall_filtered_df.to_csv(f'testing.csv', index=False)


2630

2381

,course_identifier,dept_code,school_code,course_name,term,section,call_number,instructor,start_time,end_time,days
0,AFASG4990,AFAM,SAS,AFR-AMER RESEARCH-WRITING,20263,001,12141,Rachel Grace Newman,12:10,14:00,R 12:10PM-2:00PM
1,AFASG4100,AFAM,SAS,AF-AM: PRO SEMINAR,20263,001,13210,Farah Griffin,14:10,16:00,T 2:10PM-4:00PM
2,AFASG4014,AFAM,SAS,Assurance of Infinity: In the Caribbean,20263,001,14264,Rachel Grace Newman,14:10,16:00,R 2:10PM-4:00PM
3,AFASG4010,AFAM,SAS,Reading the works of Jamaica Kincaid,20263,001,12308,Edwidge Danticat,16:10,18:00,M 4:10PM-6:00PM
4,AFASW4003,AFAM,SAS,Writing and Editing the Black Studies Journal,20263,001,14265,Jafari S Allen,16:10,18:00,T 4:10PM-6:00PM
...,...,...,...,...,...,...,...,...,...,...,...
4210,STATW4203,STAT,SAS,PROBABILITY THEORY,20263,003,14573,Banu Baydil,18:10,19:25,TR 6:10PM-7:25PM
4211,STATW4001,STAT,SAS,INTRODUCTION TO PROBABILITY AND STATISTICS,20263,001,14570,Banu Baydil,18:10,19:25,MW 6:10PM-7:25PM
4212,STATG3108,STAT,SAS,APPLIED DEEP LEARNING AND AI,20263,001,14569,Parijat Dube,18:10,19:25,TR 6:10PM-7:25PM
4213,STATW1001,STAT,SAS,INTRO TO STATISTICAL REASONING,20263,001,14548,Tian Zheng,10:10,11:25,TR 10:10AM-11:25AM


In [ ]:

# grad_pattern = r'.{4}G'
# remove = df['course.course_identifier'].str.contains(grad_pattern, regex=True, na=False)
# df_undergrad = df[~remove]
# df_undergrad

fall_filtered_df = df_fall_26[df_fall_26['school_code'].isin(desired_codes)]
display(fall_filtered_df.shape[0])
# display(fall_filtered_df)

fall_filtered_df = fall_filtered_df.dropna(subset=['days'])
fall_filtered_df = fall_filtered_df.dropna(subset=['end_time'])
fall_filtered_df = fall_filtered_df.dropna(subset=['start_time'])
display(fall_filtered_df.shape[0])

In [ ]:

# spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
# display(spring_filtered_df.shape[0])

df = fall_filtered_df.copy()
display(group_class(df, 'fall 26'))


# spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
# display(fall_filtered_df.shape[0])
# display(fall_filtered_df)
spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
spring_filtered_df = spring_filtered_df.dropna(subset=['days'])
spring_filtered_df = spring_filtered_df.dropna(subset=['end_time'])
spring_filtered_df = spring_filtered_df.dropna(subset=['start_time'])

df = spring_filtered_df.copy()
display(group_class(df, 'spring 26'))



# fall_25_filtered_df = df_fall_25[df_spring_26['school_code'].isin(desired_codes)]
# display(fall_filtered_df.shape[0])
# display(fall_filtered_df)
fall_25_filtered_df = df_fall_25[df_fall_25['school_code'].isin(desired_codes)]
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['days'])
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['end_time'])
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['start_time'])

df = fall_25_filtered_df.copy()
display(group_class(df, 'fall 25'))
